In [53]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Polygon, Point
from haversine import haversine_vector, Unit

**NUTS4-NUTS4 distance**

In [79]:
# read geography
rs_shape = gpd.read_file("../data/shape_files/nuts4_shape.geojson")


In [80]:
# centroid coords
rs_shape = rs_shape.to_crs("epsg:4326")
rs_shape["lon"] = rs_shape["geometry"].centroid.x
rs_shape["lat"] = rs_shape["geometry"].centroid.y

/var/folders/9d/8j37_fks51x11mk0_zwqsd940000gn/T/ipykernel_19483/1759052024.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  rs_shape["lon"] = rs_shape["geometry"].centroid.x
/var/folders/9d/8j37_fks51x11mk0_zwqsd940000gn/T/ipykernel_19483/1759052024.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  rs_shape["lat"] = rs_shape["geometry"].centroid.y


In [81]:
# create full region-region combination
nuts4_name = rs_shape["region_name"].unique()
full_comb = pd.MultiIndex.from_product(
    [nuts4_name, nuts4_name], names=["region1", "region2"]
)
full_comb = pd.DataFrame(index=full_comb).reset_index().sort_values(by=["region1", "region2"])

In [82]:
# add coords
full_comb = pd.merge(
    full_comb,
    rs_shape[["region_name", "lon", "lat"]].drop_duplicates(),
    left_on="region1",
    right_on="region_name",
    how="left"
)
full_comb = pd.merge(
    full_comb,
    rs_shape[["region_name", "lon", "lat"]].drop_duplicates(),
    left_on="region2",
    right_on="region_name",
    how="left",
    suffixes=["1", "2"]
)
full_comb.drop(columns=["region_name1", "region_name2"], inplace=True)

In [83]:
# distance calculation
full_comb["coords1"] = list(
    zip(full_comb["lat1"], full_comb["lon1"])
)
full_comb["coords2"] = list(
    zip(full_comb["lat2"], full_comb["lon2"])
)
full_comb["distance"] = haversine_vector(
    full_comb["coords1"].tolist(), full_comb["coords2"].tolist()
)

In [84]:
# ksh region codes
region_codes = pd.read_csv("../outputs/region_names_codes.csv", sep=";")

In [85]:
# for KSH incheck
full_comb = pd.merge(
    full_comb[["region1", "region2", "distance"]],
    region_codes[["nuts4_code", "nuts4_name"]].drop_duplicates(),
    left_on="region1",
    right_on="nuts4_name",
    how="left"
)
full_comb = pd.merge(
    full_comb,
    region_codes[["nuts4_code", "nuts4_name"]].drop_duplicates(),
    left_on="region2",
    right_on="nuts4_name",
    how="left",
    suffixes=["1", "2"]
)
full_comb.rename(columns={"nuts4_code1":"region_code1", "nuts4_code2":"region_code2"}, inplace=True)
full_comb.drop(columns=["nuts4_name1", "nuts4_name2"], inplace=True)

In [87]:
# export
full_comb.to_csv("../outputs/nuts4_nuts4_distance.csv", index=False, sep=";")